# 04 — Inferenza dei Layer Normativi e Heatmap di Ibridità

Questo notebook implementa il cuore metodologico del progetto: **i livelli gerarchici
non sono predefiniti ma emergono dai dati** della specifica materia analizzata.

## Pipeline in quattro fasi

| Fase | Input | Operazione | Output |
|---|---|---|---|
| **A** | Segmenti di ogni atto | LLM 1: descrizione funzionale contestualizzata per segmento | `segments_descriptions.csv` |
| **B** | Descrizioni funzionali | Embedding + UMAP + HDBSCAN → cluster = layer emergenti | — |
| **B.2** | 10 descrizioni representative per cluster | LLM 2: nome del layer | `layer_mapping.csv` |
| **C** | Articoli + layer noti | LLM 3: distribuzione % articolo × layer | `nodes_heatmap.csv` |
| **D** | Matrice per atto | Entropia media → score di ibridità continuo | `nodes_hybridity.csv` |

## Principio metodologico

Il clustering avviene a livello di **segmento** (articolo o considerando), non di atto.
I cluster che emergono rappresentano i livelli gerarchici specifici per quella materia —
quanti siano lo decide l'algoritmo, non l'analista.

Un atto è **ibrido** se i suoi articoli hanno distribuzioni molto diverse tra loro:
alcuni concentrati su livelli apicali, altri su livelli tecnici di dettaglio.
L'ibridità è misurata come **entropia media degli articoli**.

## Output

| File | Contenuto |
|---|---|
| `segments_descriptions.csv` | Una riga per segmento con la descrizione funzionale (LLM 1) |
| `layer_mapping.csv` | Cluster → nome layer con descrizione e rank gerarchico |
| `nodes_heatmap.csv` | Una riga per (celex, articolo) con % per ogni layer |
| `nodes_hybridity.csv` | Una riga per atto con score ibridità e layer dominante |

---

> **Nota sul costo API**: Fase A chiama l'LLM una volta per segmento, Fase C una volta
> per articolo. Con ~200 atti e ~20 segmenti/atto = ordine di 4.000–6.000 chiamate.
> Il checkpointing granulare permette di riprendere da dove si era interrotti.

## 0. Configurazione

**Modifica solo questa cella.** Il resto del notebook gira in automatico.

In [55]:
# ─────────────────────────────────────────────────────────────────────────────
#  MATERIA  →  stessa cartella scelta nei notebook 02 e 03
# ─────────────────────────────────────────────────────────────────────────────

MATERIA_NAME = "fdi_screening"   # <- modifica qui


# ── Descrizione tema per i prompt LLM ────────────────────────────────────────
# Frase in italiano che descrive la materia. Usata nei tre prompt LLM per
# contestualizzare le descrizioni funzionali.
# Esempi:
#   "controllo degli investimenti diretti esteri (FDI Screening) e poteri
#    speciali dello Stato (Golden Power) sulla sicurezza nazionale"
#   "protezione dei dati personali e privacy nel contesto digitale europeo"

TEMA_DESCRIZIONE = (
    "Foreign direct investment screening (FDI Screening)"
    "and special state powers (Golden Power) over national security"
    "and strategic infrastructure."
)


# ── Modello OpenAI ─────────────────────────────────────────────────────────────
LLM_MODEL = "gpt-5.4-mini"   # usato per tutte e tre le invocazioni LLM


# ── Parametri chiamate API ────────────────────────────────────────────────────
LLM_MAX_TOKENS_A   = 300    # Fase A: descrizioni brevi (1-2 frasi)
LLM_MAX_TOKENS_B2  = 400    # Fase B.2: JSON nome + descrizione layer
LLM_MAX_TOKENS_C   = 500    # Fase C: JSON percentuali
LLM_DELAY_SECONDS  = 0.3    # pausa tra chiamate (rispetta il rate limit)
LLM_MAX_RETRIES    = 3      # tentativi in caso di errore transitorio
LLM_RETRY_DELAY    = 5.0    # secondi tra retry


# ── Parametri checkpoint ──────────────────────────────────────────────────────
CHECKPOINT_EVERY   = 100    # segmenti/articoli tra un salvataggio e il successivo


# ── Parametri clustering ──────────────────────────────────────────────────────
EMBEDDING_MODEL     = "all-mpnet-base-v2"
UMAP_N_COMPONENTS   = 10     # dimensioni ridotte prima di HDBSCAN
UMAP_N_NEIGHBORS    = 15
UMAP_MIN_DIST       = 0.0    # 0.0 ottimizza la separazione dei cluster

HDBSCAN_MIN_CLUSTER = 25
HDBSCAN_MIN_SAMPLES = 5
N_REPR_DESCRIPTIONS = 10     # descrizioni representative per il naming

## 1. Import e Percorsi

In [3]:
import os
import json
import math
import time
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=r'C:\Users\claud\Documents\GitHub\eu-law-network-viz\.env')

from openai import OpenAI
import openai

# ── Percorsi ──────────────────────────────────────────────────────────────────
output_path  = os.path.join('..', 'data', 'output', MATERIA_NAME)
Path(output_path).mkdir(parents=True, exist_ok=True)

INPUT_FILE              = os.path.join(output_path, 'nodes_texts.csv')
EDGES_FILE              = os.path.join(output_path, 'edges_focal.csv')
SEGMENTS_DESC_FILE      = os.path.join(output_path, 'segments_descriptions.csv')
LAYER_MAPPING_FILE      = os.path.join(output_path, 'layer_mapping.csv')
NODES_HEATMAP_FILE      = os.path.join(output_path, 'nodes_heatmap.csv')
NODES_HYBRIDITY_FILE    = os.path.join(output_path, 'nodes_hybridity.csv')
HEATMAP_CKPT_FILE       = os.path.join(output_path, 'heatmap_checkpoint.csv')

print(f"Materia:       {MATERIA_NAME}")
print(f"Input:         {INPUT_FILE}")
print(f"Modello LLM:   {LLM_MODEL}")
print(f"Embedding:     {EMBEDDING_MODEL}")

Materia:       fdi_screening
Input:         ..\data\output\fdi_screening\nodes_texts.csv
Modello LLM:   gpt-5.4-mini
Embedding:     all-mpnet-base-v2


## 2. Caricamento Dati

In [5]:
nodes = pd.read_csv(INPUT_FILE)
print(f"Nodi totali: {len(nodes)}")

REQUIRED_COLS = ['Id', 'Label', 'segments', 'text_status', 'title']
missing_cols = [c for c in REQUIRED_COLS if c not in nodes.columns]
if missing_cols:
    raise RuntimeError(
        f"Colonne mancanti in {INPUT_FILE}: {missing_cols}.\n"
        "Assicurarsi che il notebook 03 sia stato eseguito completamente."
    )

nodes_ok   = nodes[nodes['text_status'] == 'ok'].copy()
nodes_fail = nodes[nodes['text_status'] != 'ok'].copy()

print(f"Atti con testo (text_status=ok): {len(nodes_ok)}")
print(f"Atti senza testo (esclusi):      {len(nodes_fail)}")
print()
print("Distribuzione text_status:")
print(nodes['text_status'].value_counts().to_string())

Nodi totali: 19
Atti con testo (text_status=ok): 19
Atti senza testo (esclusi):      0

Distribuzione text_status:
text_status
ok    19


## 3. Esplosione dei Segmenti

La colonna `segments` di ogni atto contiene una lista JSON di segmenti strutturati
(articoli, considerando, allegati). Questa cella costruisce un DataFrame flat
`segments_df` con **una riga per segmento** — l'unità di analisi del clustering.

In [6]:
def parse_segments(row):
    """Parsa la colonna 'segments' e restituisce lista di dict arricchiti."""
    celex = row.get('Label', row['Id'])
    title = str(row.get('title', ''))
    raw   = row.get('segments')

    if pd.isna(raw) or not str(raw).strip():
        return []
    try:
        segs = json.loads(str(raw))
    except (json.JSONDecodeError, ValueError):
        return []

    result = []
    for s in segs:
        result.append({
            'celex':          celex,
            'node_id':        row['Id'],
            'title_atto':     title,
            'tipo':           s.get('tipo', ''),
            'identificatore': s.get('identificatore', ''),
            'testo':          s.get('testo', ''),
        })
    return result


all_segments = []
for _, row in nodes_ok.iterrows():
    all_segments.extend(parse_segments(row))

segments_df = pd.DataFrame(all_segments)

# ID univoco per segmento
segments_df['segment_id'] = (
    segments_df['celex'] + '__' +
    segments_df['tipo'] + '__' +
    segments_df['identificatore'].astype(str)
)

# Rimuove duplicati su segment_id (stesso atto, stesso tipo, stesso identificatore)
before_dedup = len(segments_df)
segments_df = segments_df.drop_duplicates(subset='segment_id', keep='first').reset_index(drop=True)

# Rimuove segmenti con testo troppo breve per essere informativi
MIN_TESTO_LEN = 30
before = len(segments_df)
segments_df = segments_df[segments_df['testo'].str.len() >= MIN_TESTO_LEN].reset_index(drop=True)

# Esclude considerando e header preambolo — non entrano nel clustering né nella heatmap
TIPI_ESCLUSI = {'considerando', 'preambolo_header'}
before_tipi = len(segments_df)
segments_df = segments_df[~segments_df['tipo'].isin(TIPI_ESCLUSI)].reset_index(drop=True)

print(f"Segmenti estratti:              {before_dedup:,}")
print(f"Segmenti scartati (duplicati):  {before_dedup - before:,}")
print(f"Segmenti validi (>={MIN_TESTO_LEN} car): {len(segments_df):,}")
print(f"Segmenti scartati (testo):      {before - before_tipi:,}")
print(f"Segmenti scartati (tipo):       {before_tipi - len(segments_df):,}")
print()
print("Distribuzione per tipo:")
print(segments_df['tipo'].value_counts().to_string())
print()
print(f"Segmenti medi per atto:  {segments_df.groupby('celex').size().mean():.1f}")

Segmenti estratti:              1,249
Segmenti scartati (duplicati):  53
Segmenti validi (>=30 car): 848
Segmenti scartati (testo):      7
Segmenti scartati (tipo):       341

Distribuzione per tipo:
tipo
articolo    848

Segmenti medi per atto:  44.6


## 4. Fase A — Descrizione Funzionale per Segmento (LLM 1)

Per ogni segmento l'LLM produce una **descrizione funzionale contestualizzata alla materia**:
cosa fa normativamente quel segmento in questo specifico contesto.

Questo è il passaggio che rende il metodo specifico per materia: lo stesso articolo
"Definizioni" viene descritto diversamente in un regolamento FDI rispetto a uno sulla privacy.

> **Checkpoint**: i risultati vengono salvati in `segments_descriptions.csv` ogni
> `CHECKPOINT_EVERY` segmenti. Rieseguire la cella riprende dal punto di interruzione.

In [10]:
def build_prompt_functional_description(testo, tipo, identificatore, title_atto, tema):
    return f"""You are an expert in European law specialised in: {tema}.

Your task is to describe TWO dimensions of the normative role that a segment 
of a legal act plays in the regulatory pyramid.

## Dimension 1 — FUNCTION
What normative operation does this segment perform in the legal architecture?
(e.g., justifies an intervention, confers a power, sets a procedure,
defines a concept, fixes a threshold, establishes a sanction, cross-references another norm...)

## Dimension 2 — ABSTRACTION LEVEL
How general or specific is this norm relative to the overall framework?
Ask yourself: does this segment state a broad principle that governs the 
entire framework, or does it implement a narrow technical detail that 
only applies in a specific situation?
Describe this on a spectrum, for example:
- "broad foundational principle governing the entire framework"
- "structural rule defining how institutions relate to each other"
- "operational rule applying to a specific class of situations"
- "technical detail implementing a higher-level norm in a narrow case"
Do NOT use these exact phrases — infer the level from the text itself.

## Critical rules
- **Never mention the specific subject matter** (FDI, data protection, banking,
  statistics, subsidies…). The description must make sense in any regulatory domain.
- Do NOT use Lamfalussy terminology (Level 1, Level 2, etc.).
- Describe both dimensions in a single flowing description of 2-3 sentences.

## Examples of correct descriptions (note: both dimensions are present)
- "Foundational recital establishing the broad policy rationale that legitimises 
   the entire legislative intervention — operates at the highest level of 
   abstraction as it frames the purpose of the whole framework."
- "Empowerment clause delegating secondary rule-making power to the Commission 
   within defined substantive limits — sits at an intermediate level, 
   structural rather than foundational, as it organises institutional 
   competences without specifying how they must be exercised."
- "Operative provision fixing a specific numeric threshold that activates a 
   mandatory procedural obligation — highly specific, implementing a narrow 
   technical detail that only applies once a precise quantitative condition is met."
- "Commencement clause fixing the date from which the act becomes legally operative —
   procedural and terminal, applying universally to the whole act but 
   containing no substantive normative content."

## Also avoid
- Descriptions so vague they say nothing: "Sets out a provision within the framework."
- Restating what the text says without identifying the normative role.
- Mentioning only the function without any indication of abstraction level.

## Format
2-3 sentences only. No preamble, no explanation.

Act title: {title_atto}
Segment ({tipo} {identificatore}):
{testo}

Reply ONLY with the description (2-3 sentences), in English, no other text."""


def call_llm(client, prompt, max_tokens):
    """
    Chiama l'LLM OpenAI con retry automatico su errori transitori.
    Restituisce (testo_risposta, status) dove status è 'ok' | 'error' | 'empty'.
    """
    for attempt in range(LLM_MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=LLM_MODEL,
                max_completion_tokens=max_tokens,
                temperature=0.3,
                messages=[{'role': 'user', 'content': prompt}]
            )
            text = response.choices[0].message.content.strip()
            if not text:
                return '', 'empty'
            return text, 'ok'

        except openai.RateLimitError:
            wait = LLM_RETRY_DELAY * (attempt + 1) * 2
            print(f"  [RateLimit] attesa {wait:.0f}s (tentativo {attempt+1}/{LLM_MAX_RETRIES})")
            time.sleep(wait)

        except openai.APIStatusError as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

        except Exception as e:
            if attempt < LLM_MAX_RETRIES - 1:
                time.sleep(LLM_RETRY_DELAY)
            else:
                return f'ERROR: {e}', 'error'

    return 'ERROR: max retries exceeded', 'error'


print("Funzioni LLM definite.")

Funzioni LLM definite.


In [11]:
# ── TEST su 5 segmenti ────────────────────────────────────────────────────────
from openai import OpenAI

client = OpenAI()


test_segments = segments_df.sample(5, random_state=32)

for _, seg in test_segments.iterrows():
    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    
    print(f"CELEX: {seg['celex']}  |  {seg['tipo']} {seg['identificatore']}")
    print(f"Testo: {seg['testo'][:150]}...")
    print(f"→ {descrizione}")
    print()

CELEX: 32023R1441  |  articolo 4_p5
Testo: 4. The Commission may, upon written request, dispense a requesting notifying party with the obligation to provide any information in the notification ...
→ Discretionary waiver clause empowering the Commission, on written request, to relieve a notifying party from specified disclosure obligations or related formal requirements in the notification form. It operates at a fairly concrete, case-by-case level: not a general principle, but a targeted procedural flexibility mechanism that adjusts the information burden in individual proceedings.

CELEX: 32019R0452  |  articolo 9_p13
Testo: 5. A Member State shall notify the Commission and the other Member States concerned without delay, if, in exceptional circumstances, it is unable, des...
→ Operative notification rule that triggers a duty to inform the central authority and peer authorities when a state cannot secure the required information despite diligent efforts, thereby preserving transparency

In [13]:
# ── Gestione checkpoint ───────────────────────────────────────────────────────
if os.path.exists(SEGMENTS_DESC_FILE):
    segs_done = pd.read_csv(SEGMENTS_DESC_FILE)
    done_ids  = set(segs_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_ids):,} segmenti già descritti.")
else:
    segs_done = pd.DataFrame()
    done_ids  = set()
    print("Nessun checkpoint — si parte da zero.")

segments_todo = segments_df[~segments_df['segment_id'].isin(done_ids)].copy()
print(f"Da descrivere: {len(segments_todo):,}")

if len(segments_todo) == 0:
    print("✓ Tutti i segmenti già descritti — si può passare alla Fase B.")

Nessun checkpoint — si parte da zero.
Da descrivere: 848


In [14]:
%%time
from openai import OpenAI

client = OpenAI()

new_rows = []
n_ok = n_error = 0
total = len(segments_todo)

for i, (_, seg) in enumerate(segments_todo.iterrows()):

    prompt = build_prompt_functional_description(
        testo          = seg['testo'],
        tipo           = seg['tipo'],
        identificatore = seg['identificatore'],
        title_atto     = seg['title_atto'],
        tema           = TEMA_DESCRIZIONE,
    )
    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)

    descrizione, status = call_llm(client, prompt, LLM_MAX_TOKENS_A)
    if status != 'ok':
        print(f"ERRORE: {descrizione}")
        break   # blocca subito al primo errore per leggere il messaggio

    new_rows.append({
        'segment_id':              seg['segment_id'],
        'celex':                   seg['celex'],
        'node_id':                 seg['node_id'],
        'tipo':                    seg['tipo'],
        'identificatore':          seg['identificatore'],
        'testo_originale':         seg['testo'],
        'descrizione_funzionale':  descrizione,
        'llm_status':              status,
    })

    if status == 'ok':
        n_ok += 1
    else:
        n_error += 1

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([segs_done, batch], ignore_index=True) if not segs_done.empty else batch
        combined.to_csv(SEGMENTS_DESC_FILE, index=False)
        print(f"  [{i+1:>5}/{total}]  {(i+1)/total*100:5.1f}%   ok: {n_ok}   errori: {n_error}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE A — ok: {n_ok:,}   errori: {n_error:,}")
print("=" * 50)

  [  100/848]   11.8%   ok: 100   errori: 0
  [  200/848]   23.6%   ok: 200   errori: 0
  [  300/848]   35.4%   ok: 300   errori: 0
  [  400/848]   47.2%   ok: 400   errori: 0
  [  500/848]   59.0%   ok: 500   errori: 0
  [  600/848]   70.8%   ok: 600   errori: 0
  [  700/848]   82.5%   ok: 700   errori: 0
  [  800/848]   94.3%   ok: 800   errori: 0
  [  848/848]  100.0%   ok: 848   errori: 0

FASE A — ok: 848   errori: 0
CPU times: total: 9.62 s
Wall time: 34min 27s


## 5. Fase B — Embedding e Clustering → Layer Emergenti

Le descrizioni funzionali vengono embeddate con `all-mpnet-base-v2`, ridotte con
UMAP e clusterizzate con HDBSCAN. Ogni cluster che emerge è un **livello gerarchico
specifico per questa materia** — quanti ce ne sono lo decide l'algoritmo.

I segmenti assegnati al cluster `-1` (noise) vengono conservati ma esclusi dal naming.

In [15]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# Carica dal checkpoint finale
segs_desc = pd.read_csv(SEGMENTS_DESC_FILE)

segs_valid = segs_desc[
    (segs_desc['llm_status'] == 'ok') &
    segs_desc['descrizione_funzionale'].notna() &
    (segs_desc['descrizione_funzionale'].str.len() > 10)
].copy()

print(f"Segmenti con descrizione valida: {len(segs_valid):,} / {len(segs_desc):,}")
print()

# ── Embedding con cache ───────────────────────────────────────────────────────
embeddings_file = os.path.join(output_path, 'embeddings.npy')

if os.path.exists(embeddings_file):
    embeddings = np.load(embeddings_file)
    print(f"Embeddings caricati da cache: {embeddings.shape}")
else:
    print(f"Caricamento modello embedding: {EMBEDDING_MODEL} ...")
    encoder = SentenceTransformer(EMBEDDING_MODEL)
    print("Calcolo embeddings...")
    embeddings = encoder.encode(
        segs_valid['descrizione_funzionale'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    print(f"Embeddings shape: {embeddings.shape}")
    np.save(embeddings_file, embeddings)
    print(f"Embeddings salvati: {embeddings_file}")

Segmenti con descrizione valida: 848 / 848

Caricamento modello embedding: all-mpnet-base-v2 ...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calcolo embeddings...


Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Embeddings shape: (848, 768)
Embeddings salvati: ..\data\output\fdi_screening\embeddings.npy


In [16]:
import numpy as np

embeddings_file = os.path.join(output_path, 'embeddings.npy')
np.save(embeddings_file, embeddings)
print(f"Embeddings salvati: {embeddings_file}")

Embeddings salvati: ..\data\output\fdi_screening\embeddings.npy


In [53]:
print(f"UMAP: {embeddings.shape[1]}d → {UMAP_N_COMPONENTS}d ...")

reducer = umap.UMAP(
    n_components = UMAP_N_COMPONENTS,
    n_neighbors  = UMAP_N_NEIGHBORS,
    min_dist     = UMAP_MIN_DIST,
    metric       = 'cosine',
    random_state = 42,
    low_memory   = False,
)
embeddings_reduced = reducer.fit_transform(embeddings)
print(f"Shape ridotta: {embeddings_reduced.shape}")

UMAP: 768d → 10d ...


c:\Users\claud\Documents\GitHub\eu-law-network-viz\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Shape ridotta: (848, 10)


In [56]:
print(f"HDBSCAN (min_cluster={HDBSCAN_MIN_CLUSTER}, min_samples={HDBSCAN_MIN_SAMPLES}) ...")

clusterer = hdbscan.HDBSCAN(
    min_cluster_size         = HDBSCAN_MIN_CLUSTER,
    min_samples              = HDBSCAN_MIN_SAMPLES,
    cluster_selection_method = 'eom',
    prediction_data          = True,
)
cluster_labels = clusterer.fit_predict(embeddings_reduced)

segs_valid = segs_valid.copy()
segs_valid['cluster_id'] = cluster_labels
if hasattr(clusterer, 'probabilities_'):
    segs_valid['cluster_prob'] = clusterer.probabilities_

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = (cluster_labels == -1).sum()

print()
print("=" * 50)
print("RISULTATO CLUSTERING")
print("=" * 50)
print(f"  Layer trovati:          {n_clusters}")
print(f"  Segmenti noise (-1):    {n_noise:,}  ({n_noise/len(cluster_labels)*100:.1f}%)")
print()

cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
print("Distribuzione segmenti per cluster:")
for cid, cnt in cluster_counts.items():
    tag = 'NOISE' if cid == -1 else f'cluster_{cid}'
    bar = '█' * min(40, int(cnt / cluster_counts.max() * 40))
    print(f"  {tag:>12}: {cnt:>5}  {bar}")

HDBSCAN (min_cluster=25, min_samples=5) ...

RISULTATO CLUSTERING
  Layer trovati:          11
  Segmenti noise (-1):    120  (14.2%)

Distribuzione segmenti per cluster:
         NOISE:   120  ████████████████
     cluster_0:    73  ██████████
     cluster_1:    38  █████
     cluster_2:    32  ████
     cluster_3:    53  ███████
     cluster_4:   292  ████████████████████████████████████████
     cluster_5:    40  █████
     cluster_6:    27  ███
     cluster_7:    66  █████████
     cluster_8:    40  █████
     cluster_9:    38  █████
    cluster_10:    29  ███


## 6. Fase B.2 — Ordinamento Gerarchico e Naming dei Layer (LLM 2)

**Naming**: LLM 2 riceve le 10 descrizioni più rappresentative di ogni cluster e assegna un nome e una descrizione al layer,
ed infine li ordina gerarchicamente.

In [28]:
from openai import OpenAI

client = OpenAI()
   

# ── Funzioni ──────────────────────────────────────────────────────────────────

def call_llm(client, prompt, max_tokens):
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
        )
        return response.choices[0].message.content.strip(), 'ok'
    except Exception as e:
        print(f"  Errore LLM: {e}")
        return '', 'error'


def get_representative_descriptions(segs_df, cluster_id, n=N_REPR_DESCRIPTIONS):
    subset = segs_df[segs_df['cluster_id'] == cluster_id].copy()
    if 'cluster_prob' in subset.columns:
        subset = subset.nlargest(n, 'cluster_prob')
    else:
        subset = subset.sample(min(n, len(subset)), random_state=42)
    return subset['descrizione_funzionale'].tolist()


def build_prompt_layer_naming(descriptions):
    desc_list = '\n'.join([f"  {i+1}. {d}" for i, d in enumerate(descriptions)])
    return f"""You are an expert in European law.

Below are functional descriptions of legal segments that cluster together 
in a corpus analysis. They share the same normative role in the regulatory 
hierarchy.

Identify what hierarchical function unites them and give this cluster:
- a SHORT name (3-6 words, functional not thematic)
- a brief description (2-3 sentences) of the normative role

GOOD name examples:
- "Empowerment and delegation clauses"
- "Justificatory and context-setting recitals"
- "Commencement and temporal provisions"
- "Procedural safeguards and consultation requirements"
- "Scope-defining and definitional provisions"

BAD name examples (too thematic, too long):
- "Foundational purposes and cooperative architecture of EU FDI screening"
- "Confidentiality rules in foreign subsidy investigations"

Representative segments:
{desc_list}

Reply ONLY in this JSON format (no backticks):
{{"nome": "Layer Name", "descrizione": "Description in 2-3 sentences."}}"""


def build_prompt_layer_ranking(layer_records):
    layers_text = '\n'.join([
        f"  cluster_{r['cluster_id']}: {r['layer_name']} — {r['layer_description'][:120]}"
        for r in layer_records
    ])
    return f"""You are an expert in European law.

Below are functional layers found in a corpus of EU legal acts.
Rank them from most foundational (1 = constitutional basis, enabling norms, 
definitions) to most technical and operational (n = filing rules, timing, 
cross-references).

Layers:
{layers_text}

Reply ONLY with a JSON array of cluster_ids in order from most foundational 
to most technical (no other text, no backticks):
[cluster_id_1, cluster_id_2, ..., cluster_id_n]"""


# ── Fase B.2 — Naming ─────────────────────────────────────────────────────────

if os.path.exists(LAYER_MAPPING_FILE):
    layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
    layer_records = layer_mapping_df.to_dict('records')
    print(f"Checkpoint trovato: {len(layer_records)} layer già nominati e ordinati.")
else:

    valid_cluster_ids = sorted([cid for cid in set(cluster_labels) if cid != -1])
    layer_records = []

    for cluster_id in valid_cluster_ids:
        repr_descs = get_representative_descriptions(segs_valid, cluster_id)
        prompt = build_prompt_layer_naming(repr_descs)
        response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_B2)

        nome        = f'Layer_{cluster_id}'
        descrizione = ''
        if status == 'ok':
            try:
                clean   = response_text.replace('```json', '').replace('```', '').strip()
                parsed  = json.loads(clean)
                nome    = parsed.get('nome', nome)
                descrizione = parsed.get('descrizione', '')
            except json.JSONDecodeError:
                nome = response_text[:80].strip()
                print(f"  cluster_{cluster_id}: JSON non valido, uso raw text come nome")

        layer_records.append({
            'cluster_id':        cluster_id,
            'layer_rank':        None,  # assegnato dopo
            'layer_name':        nome,
            'layer_description': descrizione,
            'n_segments':        int((segs_valid['cluster_id'] == cluster_id).sum()),
            'repr_descriptions': json.dumps(repr_descs, ensure_ascii=False),
            'llm_status':        status,
        })
        print(f"  cluster_{cluster_id:>3} → '{nome}'")
        time.sleep(LLM_DELAY_SECONDS)


    # ── Fase B.3 — Ranking LLM ────────────────────────────────────────────────────

    print("\nOrdinamento gerarchico via LLM...")
    prompt_rank = build_prompt_layer_ranking(layer_records)
    response_rank, status_rank = call_llm(client, prompt_rank, 200)

    if status_rank == 'ok':
        try:
            clean        = response_rank.replace('```json', '').replace('```', '').strip()
            ordered_ids  = json.loads(clean)
            ordered_ids = [int(str(x).replace('cluster_', '')) for x in ordered_ids]
            llm_rank     = {cid: rank + 1 for rank, cid in enumerate(ordered_ids)}
            # Fallback per cluster_id non restituiti dall'LLM
            missing = [r['cluster_id'] for r in layer_records if r['cluster_id'] not in llm_rank]
            for i, cid in enumerate(missing):
                llm_rank[cid] = len(ordered_ids) + i + 1
                print(f"  Warning: cluster_{cid} non nel ranking LLM, appeso in fondo")
        except (json.JSONDecodeError, TypeError) as e:
            print(f"  Ranking LLM non valido ({e}) — fallback su dimensione cluster")
            sorted_by_size = sorted(layer_records, key=lambda r: r['n_segments'], reverse=True)
            llm_rank = {r['cluster_id']: rank + 1 for rank, r in enumerate(sorted_by_size)}
    else:
        print("  Ranking LLM fallito — fallback su dimensione cluster")
        sorted_by_size = sorted(layer_records, key=lambda r: r['n_segments'], reverse=True)
        llm_rank = {r['cluster_id']: rank + 1 for rank, r in enumerate(sorted_by_size)}

    for r in layer_records:
        r['layer_rank'] = llm_rank[r['cluster_id']]


# ── Salvataggio e stampa ──────────────────────────────────────────────────────

layer_mapping_df = pd.DataFrame(layer_records).sort_values('layer_rank')
layer_mapping_df.to_csv(LAYER_MAPPING_FILE, index=False)

print()
print("=" * 60)
print("LAYER TROVATI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    print(f"  [{row['layer_rank']}] {row['layer_name']}")
    print(f"      {str(row['layer_description'])[:120]}")
    print(f"      N segmenti: {row['n_segments']:,}")
    print()

  cluster_  0 → 'Incorporation by reference clauses'
  cluster_  1 → 'Empowering and enabling provisions'
  cluster_  2 → 'Delegation and empowerment clauses'
  cluster_  3 → 'Implementation and reporting duties'
  cluster_  4 → 'Scope and applicability provisions'
  cluster_  5 → 'Operational procedural provisions'
  cluster_  6 → 'Scope-limiting exceptions'
  cluster_  7 → 'Operational flexibility and derogation clauses'
  cluster_  8 → 'Procedural safeguards and confidentiality'
  cluster_  9 → 'Procedural notification and reporting clauses'
  cluster_ 10 → 'Procedural coordination and consultation clauses'

Ordinamento gerarchico via LLM...

LAYER TROVATI
  [1] Scope and applicability provisions
      These provisions delimit when and to whom the act or rule set applies, often by defining covered subjects, domains, acti
      N segmenti: 292

  [2] Empowering and enabling provisions
      These provisions grant a specific institution or Member State a conditional legal power to act

## 7. Fase C — Distribuzione Percentuale per Articolo (LLM 3)

Con i layer noti, per ogni **articolo** di ogni atto l'LLM produce una distribuzione
percentuale del contenuto tra i layer emersi.

Il risultato per ogni atto è la matrice **articoli × layer** (valori = %) che alimenta
la heatmap nell'applicazione.

> **Perché solo gli articoli?** I considerando hanno funzione giustificativa e retorica:
> spesso coprono più livelli intenzionalmente per costruire l'argomentazione legale.
> La varianza dei considerando riflette struttura retorica, non patologia.
> Gli articoli hanno funzione prescrittiva — la loro ibridità è il segnale diagnostico.

In [29]:
# Ricarica layer mapping (può essere eseguita anche senza rieseguire B)
layer_mapping_df = pd.read_csv(LAYER_MAPPING_FILE)
layer_list = [
    {
        'rank':        int(row['layer_rank']),
        'name':        row['layer_name'],
        'description': row['layer_description'],
    }
    for _, row in layer_mapping_df.sort_values('layer_rank').iterrows()
]
layer_names = [l['name'] for l in layer_list]

# Colonne CSV safe (senza spazi/slash)
def to_col(name):
    return 'pct__' + name.replace(' ', '_').replace('/', '_')[:50]

pct_cols     = [to_col(n) for n in layer_names]
col_to_layer = {to_col(n): n for n in layer_names}

# Solo gli articoli
articles_df = segments_df[segments_df['tipo'] == 'articolo'].copy()

print(f"Layer trovati: {len(layer_list)}")
for l in layer_list:
    print(f"  [{l['rank']}] {l['name']}")
print()
print(f"Articoli da classificare: {len(articles_df):,}")
print(f"Atti coinvolti:           {articles_df['celex'].nunique():,}")

Layer trovati: 11
  [1] Scope and applicability provisions
  [2] Empowering and enabling provisions
  [3] Delegation and empowerment clauses
  [4] Scope-limiting exceptions
  [5] Incorporation by reference clauses
  [6] Procedural coordination and consultation clauses
  [7] Procedural safeguards and confidentiality
  [8] Operational flexibility and derogation clauses
  [9] Procedural notification and reporting clauses
  [10] Implementation and reporting duties
  [11] Operational procedural provisions

Articoli da classificare: 848
Atti coinvolti:           19


In [30]:
def build_prompt_percentage_distribution(testo, identificatore, layer_list, tema):
    layers_desc = '\n'.join([
        f"  {l['rank']}. {l['name']}: {l['description']}"
        for l in layer_list
    ])
    layer_keys = ', '.join([f'"{l["name"]}"' for l in layer_list])

    return f"""You are an expert in European law.

Read the article below and distribute its content as a percentage across 
the following hierarchical normative layers. Each layer represents a distinct 
functional role in the regulatory hierarchy — assign percentages based on 
what normative function each part of the article performs, not on its topic.

Layers:
{layers_desc}

Rules:
- Percentages must sum to exactly 100.
- Assign 0 to layers not present in the article.
- If the article mixes layers, reflect the actual proportions.

Article {identificatore}:
{testo}

Reply ONLY with valid JSON (no other text, no backticks):
{{{layer_keys}}}"""


def parse_percentage_response(response_text, layer_names):
    """
    Parsa la risposta JSON e normalizza a somma 100.
    Restituisce None se il parsing fallisce.
    """
    try:
        clean  = response_text.replace('```json', '').replace('```', '').strip()
        parsed = json.loads(clean)
    except json.JSONDecodeError:
        return None

    values = {name: float(parsed.get(name, 0)) for name in layer_names}
    total  = sum(values.values())
    if total <= 0:
        return None
    if abs(total - 100) > 5:   # normalizza se la somma si discosta
        values = {k: v / total * 100 for k, v in values.items()}
    return values


print("Funzioni Fase C definite.")

Funzioni Fase C definite.


In [34]:
%%time
# ── Gestione checkpoint ────────────────────────────────────────────────────────
if os.path.exists(HEATMAP_CKPT_FILE):
    heatmap_done = pd.read_csv(HEATMAP_CKPT_FILE)
    done_seg_ids = set(heatmap_done['segment_id'])
    print(f"Checkpoint trovato: {len(done_seg_ids):,} articoli già classificati.")
else:
    heatmap_done = pd.DataFrame()
    done_seg_ids = set()
    print("Nessun checkpoint heatmap — si parte da zero.")

articles_todo = articles_df[~articles_df['segment_id'].isin(done_seg_ids)].copy()
print(f"Articoli da classificare: {len(articles_todo):,}")
print()

new_rows = []
n_ok_c = n_error_c = 0
total_c = len(articles_todo)

for i, (_, art) in enumerate(articles_todo.iterrows()):

    prompt = build_prompt_percentage_distribution(
        testo          = art['testo'],
        identificatore = art['identificatore'],
        layer_list     = layer_list,
        tema           = TEMA_DESCRIZIONE,
    )
    response_text, status = call_llm(client, prompt, LLM_MAX_TOKENS_C)

    distribution = None
    if status == 'ok':
        distribution = parse_percentage_response(response_text, layer_names)
        if distribution is None:
            status = 'parse_error'

    row = {
        'segment_id':  art['segment_id'],
        'celex':       art['celex'],
        'node_id':     art['node_id'],
        'articolo_id': art['identificatore'],
        'llm_status':  status,
    }
    for col, name in zip(pct_cols, layer_names):
        row[col] = round(distribution[name], 2) if distribution else 0.0

    if distribution:
        n_ok_c += 1
    else:
        n_error_c += 1

    new_rows.append(row)

    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == total_c:
        batch    = pd.DataFrame(new_rows)
        combined = pd.concat([heatmap_done, batch], ignore_index=True) if not heatmap_done.empty else batch
        combined.to_csv(HEATMAP_CKPT_FILE, index=False)
        print(f"  [{i+1:>5}/{total_c}]  {(i+1)/total_c*100:5.1f}%   ok: {n_ok_c}   errori: {n_error_c}")

    time.sleep(LLM_DELAY_SECONDS)

print()
print("=" * 50)
print(f"FASE C — ok: {n_ok_c:,}   errori: {n_error_c:,}")
print("=" * 50)

Nessun checkpoint heatmap — si parte da zero.
Articoli da classificare: 848

  [  100/848]   11.8%   ok: 100   errori: 0
  [  200/848]   23.6%   ok: 200   errori: 0
  [  300/848]   35.4%   ok: 299   errori: 1
  [  400/848]   47.2%   ok: 398   errori: 2
  [  500/848]   59.0%   ok: 498   errori: 2
  [  600/848]   70.8%   ok: 598   errori: 2
  [  700/848]   82.5%   ok: 697   errori: 3
  [  800/848]   94.3%   ok: 797   errori: 3
  [  848/848]  100.0%   ok: 845   errori: 3

FASE C — ok: 845   errori: 3
CPU times: total: 5.77 s
Wall time: 18min 10s


In [35]:
# Salva heatmap finale
heatmap_final = pd.read_csv(HEATMAP_CKPT_FILE)
heatmap_final.to_csv(NODES_HEATMAP_FILE, index=False)

print(f"Salvato: {NODES_HEATMAP_FILE}")
print(f"Righe: {len(heatmap_final):,}  |  Colonne pct: {pct_cols}")
print()

ok_mask = heatmap_final['llm_status'] == 'ok'
print("Distribuzione media % per layer (articoli ok):")
for col in pct_cols:
    mean_pct = heatmap_final.loc[ok_mask, col].mean()
    bar = '█' * int(mean_pct / 2)
    print(f"  {col_to_layer[col][:45]:.<46} {mean_pct:5.1f}%  {bar}")

Salvato: ..\data\output\fdi_screening\nodes_heatmap.csv
Righe: 848  |  Colonne pct: ['pct__Scope_and_applicability_provisions', 'pct__Empowering_and_enabling_provisions', 'pct__Delegation_and_empowerment_clauses', 'pct__Scope-limiting_exceptions', 'pct__Incorporation_by_reference_clauses', 'pct__Procedural_coordination_and_consultation_clauses', 'pct__Procedural_safeguards_and_confidentiality', 'pct__Operational_flexibility_and_derogation_clauses', 'pct__Procedural_notification_and_reporting_clauses', 'pct__Implementation_and_reporting_duties', 'pct__Operational_procedural_provisions']

Distribuzione media % per layer (articoli ok):
  Scope and applicability provisions............  15.8%  ███████
  Empowering and enabling provisions............  13.0%  ██████
  Delegation and empowerment clauses............   5.5%  ██
  Scope-limiting exceptions.....................  13.3%  ██████
  Incorporation by reference clauses............  14.2%  ███████
  Procedural coordination and consultatio

## 8. Fase D — Score di Ibridità per Atto

Lo score di ibridità misura quanto gli articoli di un atto variano nel loro livello
gerarchico. Un atto **puro** ha tutti gli articoli concentrati sullo stesso layer.
Un atto **ibrido** ha articoli che spaziano su layer molto diversi.

**Metrica: entropia di Shannon normalizzata per articolo**

$$H(a) = -\sum_{l} p_{al} \log_2(p_{al} + \varepsilon)$$

Lo score dell'atto è la **media delle entropie dei propri articoli**.
Zero = tutti gli articoli sono monofunzionali. Uno = distribuzione uniforme su tutti i layer.

In [36]:
def entropy_norm(row, pct_cols):
    """Entropia di Shannon normalizzata (0=puro, 1=uniforme)."""
    eps   = 1e-9
    probs = np.array([float(row.get(c, 0)) for c in pct_cols]) / 100.0
    probs = np.clip(probs, 0, 1)
    s     = probs.sum()
    if s < eps:
        return 0.0
    probs = probs / s
    raw   = -np.sum(probs * np.log2(probs + eps))
    maxH  = math.log2(len(pct_cols)) if len(pct_cols) > 1 else 1.0
    return float(raw / maxH)


def dominant_layer(row, pct_cols, col_to_layer):
    best = max(pct_cols, key=lambda c: float(row.get(c, 0)))
    return col_to_layer.get(best, best)


# Solo articoli classificati correttamente
heatmap_ok = heatmap_final[heatmap_final['llm_status'] == 'ok'].copy()
heatmap_ok['entropy']       = heatmap_ok.apply(lambda r: entropy_norm(r, pct_cols), axis=1)
heatmap_ok['dominant_layer'] = heatmap_ok.apply(lambda r: dominant_layer(r, pct_cols, col_to_layer), axis=1)

# ── Aggregazione per atto ──────────────────────────────────────────────────────
def agg_atto(group):
    return pd.Series({
        'hybridity_score':    group['entropy'].mean(),
        'hybridity_std':      group['entropy'].std(),
        'hybridity_max':      group['entropy'].max(),
        'n_articles':         len(group),
        'dominant_layer':     group['dominant_layer'].mode().iloc[0] if len(group) > 0 else '',
        'dominant_layer_pct': (group['dominant_layer'].value_counts().iloc[0] / len(group) * 100
                               if len(group) > 0 else 0.0),
        'most_hybrid_article': (group.nlargest(1, 'entropy')['articolo_id'].iloc[0]
                                if len(group) > 0 else ''),
    })

hybridity_df = heatmap_ok.groupby('celex').apply(agg_atto).reset_index()

# Aggiunge metadati dal nodo originale
meta_cols = [c for c in ['Id', 'Label', 'title', 'LegalType', 'Year', 'PipelineLevel']
             if c in nodes.columns]
nodes_meta = nodes[meta_cols].copy()
nodes_meta = nodes_meta.rename(columns={'Label': 'celex'}) if 'Label' in nodes_meta.columns else nodes_meta

hybridity_df = hybridity_df.merge(nodes_meta, on='celex', how='left')
hybridity_df = hybridity_df.drop_duplicates(subset=['celex'], keep='first') 
hybridity_df = hybridity_df.sort_values('hybridity_score', ascending=False)
hybridity_df.to_csv(NODES_HYBRIDITY_FILE, index=False)

print(f"Salvato: {NODES_HYBRIDITY_FILE}")
print(f"Atti analizzati: {len(hybridity_df):,}")
print()
desc = hybridity_df['hybridity_score'].describe()
print("Statistiche score ibridità:")
print(f"  Media:   {desc['mean']:.4f}")
print(f"  Mediana: {desc['50%']:.4f}")
print(f"  Max:     {desc['max']:.4f}")
print(f"  Std:     {desc['std']:.4f}")

Salvato: ..\data\output\fdi_screening\nodes_hybridity.csv
Atti analizzati: 19

Statistiche score ibridità:
  Media:   0.1760
  Mediana: 0.1993
  Max:     0.4069
  Std:     0.1132


## 9. Diagnostica e Verifica Qualità

In [37]:
print("=" * 60)
print("LAYER EMERSI")
print("=" * 60)
for _, row in layer_mapping_df.iterrows():
    n    = row['n_segments']
    pct  = n / len(segs_valid) * 100
    bar  = '█' * int(pct)
    print(f"[{row['layer_rank']:>2}] {row['layer_name']}")
    print(f"     Segmenti: {n:,}  ({pct:.1f}%)  {bar}")
    print(f"     {str(row['layer_description'])[:110]}")
    print()

LAYER EMERSI
[ 1] Scope and applicability provisions
     Segmenti: 292  (34.4%)  ██████████████████████████████████
     These provisions delimit when and to whom the act or rule set applies, often by defining covered subjects, dom

[ 2] Empowering and enabling provisions
     Segmenti: 38  (4.5%)  ████
     These provisions grant a specific institution or Member State a conditional legal power to act, usually subjec

[ 3] Delegation and empowerment clauses
     Segmenti: 32  (3.8%)  ███
     These provisions allocate regulatory or implementing power to the Commission or another institution, allowing 

[ 4] Scope-limiting exceptions
     Segmenti: 27  (3.2%)  ███
     These provisions function as narrow carve-outs that qualify, limit, or suspend the operation of a general rule

[ 5] Incorporation by reference clauses
     Segmenti: 73  (8.6%)  ████████
     These provisions do not usually establish standalone substantive rules; instead, they hook an internal rule to

[ 6] Procedural c

In [38]:
print("=" * 60)
print("TOP 10 ATTI PIÙ IBRIDI")
print("=" * 60)
print()
for _, row in hybridity_df.head(10).iterrows():
    celex_label = row.get('Label', row.get('celex', ''))
    print(f"  {row['hybridity_score']:.4f}  {celex_label}")
    print(f"           Layer dom: {row['dominant_layer']} ({row['dominant_layer_pct']:.0f}% art.)")
    print(f"           Art: {row['n_articles']}  |  Più ibrido: {row['most_hybrid_article']}")
    print(f"           {str(row.get('title',''))[:70]}")
    print()

TOP 10 ATTI PIÙ IBRIDI

  0.4069  32014R0228
           Layer dom: Empowering and enabling provisions (50% art.)
           Art: 2  |  Più ibrido: 1
           COMMISSION IMPLEMENTING REGULATION (EU) No 228/2014 of 10 March 2014 a

  0.3240  32023R1472
           Layer dom: Implementation and reporting duties (50% art.)
           Art: 2  |  Più ibrido: 1
           COMMISSION IMPLEMENTING REGULATION (EU) 2023/1472 of 17 July 2023 amen

  0.2721  32005R0184
           Layer dom: Implementation and reporting duties (29% art.)
           Art: 21  |  Più ibrido: 7
           REGULATION (EC) No 184/2005 OF THE EUROPEAN PARLIAMENT AND OF THE COUN

  0.2669  32020D0787
           Layer dom: Operational procedural provisions (50% art.)
           Art: 2  |  Più ibrido: 1
           COMMISSION DECISION (EU) 2020/787 of 16 June 2020 extending the transi

  0.2394  32023R1441
           Layer dom: Operational procedural provisions (26% art.)
           Art: 147  |  Più ibrido: 5_p1
           CO

In [40]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Istogramma ibridità
ax1 = axes[0]
ax1.hist(hybridity_df['hybridity_score'], bins=20, edgecolor='black', color='steelblue')
ax1.axvline(hybridity_df['hybridity_score'].median(), color='red', linestyle='--',
            label=f"Mediana = {hybridity_df['hybridity_score'].median():.3f}")
ax1.set_xlabel('Hybridity Score')
ax1.set_ylabel('N atti')
ax1.set_title('Distribuzione Score di Ibridità')
ax1.legend()

# Top 15 atti
ax2 = axes[1]
top15  = hybridity_df.head(15)
labels = top15['celex'].apply(lambda x: str(x)[:14]).tolist()
ax2.barh(range(len(top15)), top15['hybridity_score'], color='tomato')
ax2.set_yticks(range(len(top15)))
ax2.set_yticklabels(labels, fontsize=8)
ax2.invert_yaxis()
ax2.set_xlabel('Hybridity Score')
ax2.set_title('Top 15 Atti più Ibridi')

plt.tight_layout()

fig_dir  = os.path.join(output_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'hybridity_distribution.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figura salvata: {fig_path}")

Figura salvata: ..\data\output\fdi_screening\figures\hybridity_distribution.png


C:\Users\claud\AppData\Local\Temp\ipykernel_28528\3557925531.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

# ── Seleziona l'atto ──────────────────────────────────────────────────────────
CELEX_TARGET = '32019R0452'  # FDI Screening Regulation

atto = heatmap_final[
    (heatmap_final['celex'] == CELEX_TARGET) &
    (heatmap_final['llm_status'] == 'ok')
].copy()

# Aggrega per articolo (media delle percentuali tra i segmenti)
atto['articolo_base'] = atto['articolo_id'].str.extract(r'^(\d+)')[0]

atto_agg = atto.groupby('articolo_base')[pct_cols].mean()

# Ordina numericamente
atto_agg.index = atto_agg.index.astype(int)
atto_agg = atto_agg.sort_index()
atto_agg.index = atto_agg.index.astype(str)

matrix      = atto_agg.values / 100.0
article_ids = atto_agg.index.tolist()

# Nomi brevi per gli assi
layer_names  = [col_to_layer.get(c, c) for c in pct_cols]

# Nomi layer abbreviati per leggibilità
short_names = [n.replace(' and ', '\n& ').replace(' clauses', '').replace(' provisions', '')
               .replace(' rules', '').replace(' conditions', '') for n in layer_names]

# ── Plot ──────────────────────────────────────────────────────────────────────
fig_w = max(14, len(layer_names) * 1.2)
fig_h = max(8,  len(article_ids) * 0.35)

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

cmap = plt.cm.Blues
im   = ax.imshow(matrix, aspect='auto', cmap=cmap, vmin=0, vmax=1,
                 interpolation='nearest')

# Etichette celle 
for i in range(len(article_ids)):
    for j in range(len(pct_cols)):
        val = matrix[i, j]
        color = 'white' if val > 0.5 else 'black'
        label = f'{val*100:.0f}%'  # mostra sempre, anche 0%
        ax.text(j, i, label, ha='center', va='center',
                fontsize=7, color=color)

# Assi
ax.set_xticks(range(len(short_names)))
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(article_ids)))
ax.set_yticklabels(article_ids, fontsize=7)

ax.set_xlabel('Normative Layer', fontsize=10, labelpad=10)
ax.set_ylabel('Article', fontsize=10)
ax.set_title(f'Heatmap — {CELEX_TARGET}\nPer-article normative layer distribution',
             fontsize=12, pad=12)

plt.colorbar(im, ax=ax, label='% content in layer', shrink=0.6)
plt.tight_layout()

fig_dir  = os.path.join(output_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, f'heatmap_{CELEX_TARGET}.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print("Salvato.")

Salvato.


C:\Users\claud\AppData\Local\Temp\ipykernel_28528\3324422440.py:70: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9b. Heatmap Aggregata — Tutti gli Atti

Una singola heatmap con **tutti gli atti** sull'asse Y e i **layer normativi** sull'asse X.

Ogni cella mostra la **percentuale media** del contenuto dell'atto in quel layer,
calcolata come media delle distribuzioni di tutti gli articoli dell'atto.

Gli atti sono ordinati per **score di ibridità decrescente** (l'atto più ibrido in cima).

In [42]:
import matplotlib.pyplot as plt
import numpy as np

# ── 1. Aggregazione: media pct per atto ───────────────────────────────────────
act_means = (
    heatmap_ok
    .groupby('celex')[pct_cols]
    .mean()
    .reset_index()
)

act_means = act_means.merge(
    hybridity_df[['celex', 'hybridity_score', 'n_articles']],
    on='celex', how='left'
)

# Ordina per hybridity_score decrescente (più ibrido in cima)
act_means = act_means.sort_values('hybridity_score', ascending=False).reset_index(drop=True)

# ── 2. Matrice e label ────────────────────────────────────────────────────────
matrix     = act_means[pct_cols].values / 100.0
act_ids    = act_means['celex'].tolist()
hyb_scores = act_means['hybridity_score'].tolist()
n_articles = act_means['n_articles'].tolist()

layer_names = [col_to_layer.get(c, c) for c in pct_cols]
short_names = [
    n.replace(' and ', '\n& ')
     .replace(' clauses', '')
     .replace(' provisions', '')
     .replace(' rules', '')
     .replace(' conditions', '')
    for n in layer_names
]

y_labels = [
    f"{celex}  (H={h:.3f}, n={int(n)})"
    for celex, h, n in zip(act_ids, hyb_scores, n_articles)
]

# ── 3. Plot ───────────────────────────────────────────────────────────────────
n_acts   = len(act_ids)
n_layers = len(pct_cols)

fig, ax = plt.subplots(figsize=(max(18, n_layers * 1.5), max(8, n_acts * 0.55)))

im = ax.imshow(matrix, aspect='auto', cmap=plt.cm.Blues, vmin=0, vmax=1,
               interpolation='nearest')

# Annotazioni celle
for i in range(n_acts):
    for j in range(n_layers):
        val   = matrix[i, j]
        color = 'white' if val > 0.50 else ('lightgray' if val < 0.05 else 'black')
        ax.text(j, i, f'{val*100:.0f}%', ha='center', va='center',
                fontsize=6.5, color=color,
                fontweight='bold' if val > 0.35 else 'normal')

# Linee divisorie
for y in range(1, n_acts):
    ax.axhline(y - 0.5, color='white', linewidth=1.0 if y % 5 == 0 else 0.3)
for x in range(1, n_layers):
    ax.axvline(x - 0.5, color='white', linewidth=0.3)

# Assi
ax.set_xticks(range(n_layers))
ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(n_acts))
ax.set_yticklabels(y_labels, fontsize=8, family='monospace')
ax.set_xlabel('Normative Layer  (emerged from data)', fontsize=10, labelpad=12)
ax.set_ylabel('Legal Act  (sorted by hybridity score ↓)', fontsize=10, labelpad=10)
ax.set_title(
    'Aggregate Heatmap — All 19 Acts × 13 Normative Layers\n'
    'Mean % of articles per layer  |  FDI Screening corpus',
    fontsize=12, pad=14
)

cb = plt.colorbar(im, ax=ax, label='Mean % content in layer', shrink=0.55, pad=0.02)
cb.ax.tick_params(labelsize=8)

plt.tight_layout()

fig_dir  = os.path.join(output_path, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, 'heatmap_all_acts.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Salvato: {fig_path}")
print(f"Atti: {n_acts}  |  Layer: {n_layers}  |  Matrice: {matrix.shape}")

Salvato: ..\data\output\fdi_screening\figures\heatmap_all_acts.png
Atti: 19  |  Layer: 11  |  Matrice: (19, 11)


C:\Users\claud\AppData\Local\Temp\ipykernel_28528\4083319636.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Riepilogo Output

Verifica che tutti i file di output siano stati prodotti correttamente.

In [43]:
output_files = {
    'segments_descriptions.csv': SEGMENTS_DESC_FILE,
    'layer_mapping.csv':         LAYER_MAPPING_FILE,
    'nodes_heatmap.csv':         NODES_HEATMAP_FILE,
    'nodes_hybridity.csv':       NODES_HYBRIDITY_FILE,
}

print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)

all_ok = True
for name, path in output_files.items():
    if os.path.exists(path):
        size_kb = os.path.getsize(path) / 1024
        df_tmp  = pd.read_csv(path)
        print(f"  ✓ {name}")
        print(f"    Righe: {len(df_tmp):,}  |  Dim: {size_kb:.1f} KB")
        print(f"    Colonne: {list(df_tmp.columns)[:6]}{'...' if len(df_tmp.columns) > 6 else ''}")
    else:
        print(f"  ✗ {name} — FILE MANCANTE")
        all_ok = False
    print()

if all_ok:
    print("✓ Pipeline 04 completata.")
else:
    print("  Alcuni file mancano — rieseguire le celle corrispondenti.")

OUTPUT FILES
  ✓ segments_descriptions.csv
    Righe: 848  |  Dim: 606.3 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'tipo', 'identificatore', 'testo_originale']...

  ✓ layer_mapping.csv
    Righe: 11  |  Dim: 47.9 KB
    Colonne: ['cluster_id', 'layer_rank', 'layer_name', 'layer_description', 'n_segments', 'repr_descriptions']...

  ✓ nodes_heatmap.csv
    Righe: 848  |  Dim: 88.0 KB
    Colonne: ['segment_id', 'celex', 'node_id', 'articolo_id', 'llm_status', 'pct__Scope_and_applicability_provisions']...

  ✓ nodes_hybridity.csv
    Righe: 19  |  Dim: 8.1 KB
    Colonne: ['celex', 'hybridity_score', 'hybridity_std', 'hybridity_max', 'n_articles', 'dominant_layer']...

✓ Pipeline 04 completata.
